# Kidney Stones Dataset Preprocessing

This notebook is dedicated to preprocess the given dataset into a required format/structure and divide it into train/val/test splits.


# Google Colab only

### Download required packages

In [ ]:
!pip install -r https://raw.githubusercontent.com/pmalesa/lesion_detector/main/notebooks/requirements.txt

### Mount DeepLesion images and checkpoints from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content

# remove existing link if any
!rm -rf data/kidney_stones

!mkdir -p data
!ln -s /content/drive/MyDrive/datasets/kidney_stones/data/kidney_stones data/kidney_stones
!ls -l data

# Import all packages

In [ ]:
# General packages
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from typing import Any
from PIL import Image
import shutil
import random
from pathlib import Path
import yaml
import cv2

# Image preprocessing and utility methods

In [ ]:
def normalize(img: NDArray[np.uint16], per_image_norm: bool):
    """
    Normalizes the input image
    """

    img = img.astype(np.float32)
    if not per_image_norm:
        return img / 65535.0
    max = np.max(img)
    min = np.min(img)
    img = (img - min) / (max - min)
    return img

def convert_to_hu(img: NDArray[np.uint16], norm: bool, hu_min=-1024, hu_max=3071):
    """
    Converts the pixel data of a uint16
    CT image to Hounsfield Units (HU).
    """

    hu_img = img.astype(np.int32) - 32768
    hu_img = np.clip(hu_img, hu_min, hu_max).astype(np.float32)
    if norm:
        hu_img = (hu_img - hu_min) / (hu_max - hu_min)
        hu_img = np.clip(hu_img, 0.0, 1.0)
    return hu_img

def load_image(path: str, hu_scale: bool = True, norm: bool = True, per_image_norm: bool = True):
    """
    Loads an image given its path
    and returns it as a numpy array.
    """
    
    img = Image.open(path)
    img_array = np.array(img)
    if hu_scale:
        hu_min = -160
        hu_max = 240
        return convert_to_hu(img_array, norm, hu_min, hu_max)
    elif norm:
        return normalize(img_array, per_image_norm)
    return img_array

def save_image(img_array: NDArray[Any], path: str):
    """
    Saves the input image to the specified path.
    """

    if img_array.dtype != np.uint16:
        img_array = img_array.astype(np.uint16)
    img = Image.fromarray(img_array)
    img.save(path)

def save_normalized_image_uint8(img_array: NDArray[Any], path: str):
    """
    Saves the normalized input image data (0.0 - 1.0) 
    to the specified path with uint8 precision (0 - 255).
    """

    img_array_scaled = (img_array * 255.0).clip(0, 255).astype(np.uint8)
    img = Image.fromarray(img_array_scaled)
    img.save(path)

def show_image(img: NDArray[np.float32], title="Example Image", cmap="gray"):
    """
    Shows the image given its data, title and colour map.
    """

    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap=cmap)
    if title is not None:
        plt.title(title)
    plt.axis("off")
    plt.show()


# Set paths to kidney stones images and metadata

## Paths to unprocessed data

In [ ]:
# Google Colab
kidney_stones_images_path_unprocessed = Path("data/kidney_stones/")
kidney_stones_labels_path_unprocessed = Path("data/kidney_stones/")

In [ ]:
# Local
kidney_stones_images_path_unprocessed = Path("../data/kidney_stones/")
kidney_stones_labels_path_unprocessed = Path("../data/kidney_stones/")

## Paths to processed data

In [ ]:
kidney_stones_data_dir = Path("data/kidney_stones/")
kidney_stones_images_path_processed = kidney_stones_data_dir / "kidney_stones_processed_uint8/images"
kidney_stones_labels_path_processed = kidney_stones_data_dir / "kidney_stones_processed_uint8/labels"

## Paths to target split directory

In [ ]:
current_seed = 42   #   [42, 314. 666]

In [ ]:
seed_split_map = {
    42: "1",
    314: "2",
    666: "3"
}

target_yolo_dir = kidney_stones_data_dir / f"kidney_stones_yolo_split_{seed_split_map[current_seed]}"
target_fasterrcnn_dir = kidney_stones_data_dir / f"kidney_stones_fasterrcnn_split_{seed_split_map[current_seed]}"

IMAGE_SIZE = 512
random.seed(current_seed)

print(f"INFO: Current seed: {current_seed} / Current split: {seed_split_map[current_seed]}")

# Process the kidney stones dataset

In [ ]:
'''
Preprocess the kidney stones dataset by:
    1) Load all the images and labels from original kidney stones dataset (initially divided into train/val/test split),
    2) Resize each image to 512x512 uint8 grayscale PNG format (original images were JPGs),
    3) Save each image to "kidney_stones_processed_uint8/images/images_xxxx.png" path,
    4) Save each label in cxcywh normalized bbox format to "kidney_stones_processed_uint8/labels/label_xxxx.txt" (original format was cxcywh normalized already),
    5) Result processed kidney stone dataset layout:

        |-kidney_stones_processed_uint8
        |----images
            |----image_0001.png
            |----image_0002.png
                      (...)
        |----labels
            |----image_0001.png
            |----image_0001.png
                      (...)
'''

# Clear the existing directory with preprocessed images
if kidney_stones_images_path_processed.exists() and kidney_stones_images_path_processed.is_dir():
    shutil.rmtree(kidney_stones_images_path_processed)
kidney_stones_images_path_processed.mkdir(parents=True, exist_ok=True)

if kidney_stones_labels_path_processed.exists() and kidney_stones_labels_path_processed.is_dir():
    shutil.rmtree(kidney_stones_labels_path_processed)
kidney_stones_labels_path_processed.mkdir(parents=True, exist_ok=True)

(kidney_stones_images_path_processed).mkdir(parents=True, exist_ok=True)
(kidney_stones_labels_path_processed).mkdir(parents=True, exist_ok=True)

# 1) Collect all samles (image + label)
samples = []
for split in ["train", "valid", "test"]:
    image_dir = kidney_stones_images_path_unprocessed / split / "images"
    label_dir = kidney_stones_labels_path_unprocessed / split / "labels"

    if not image_dir.exists():
        continue

    for image_path in image_dir.glob("*"):
        label_path = label_dir / (image_path.stem + ".txt")
        if label_path.exists():
            samples.append((image_path, label_path))

print(f"Total (image, label) samples found: {len(samples)}")

# 2) Load each image save it in 512x512 grayscale PNG format (+ rename)
for i, (unprocessed_image_path, unprocessed_label_path) in enumerate(samples):
    image_str_number = f"{i + 1:05d}"
    processed_image_path = kidney_stones_images_path_processed / f"image_{image_str_number}.png"
    processed_label_path = kidney_stones_labels_path_processed / f"image_{image_str_number}.txt"

    image = Image.open(unprocessed_image_path).convert("L") # Convert to grayscale (1 channel)
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)

    image.save(processed_image_path, format="PNG")
    shutil.copy(unprocessed_label_path, processed_label_path)

    print(f"\rProcessed: {i + 1} / {len(samples)}", end="", flush=True)

# 3) Verify the result
for i in range(len(samples)):
    image_str_number = f"{i + 1:05d}"
    processed_image_path = kidney_stones_images_path_processed / f"image_{image_str_number}.png"
    processed_label_path = kidney_stones_labels_path_processed / f"image_{image_str_number}.txt"

    image = Image.open(processed_image_path)
    arr = np.array(image)
    if arr.shape != (512, 512) or arr.dtype != np.uint8:
        raise ValueError(f"ERROR: Not all images were processed successfully!")
    print(f"\rVerified: {i + 1} / {len(samples)}", end="", flush=True)
    
print(f"\nSUCCES: All images were preprocessed correctly.")

# Create a single train/val/test data split

## YOLO format

In [ ]:
# deeplesion_metadata_preprocessed = load_metadata(deeplesion_preprocessed_metadata_path)

# # Source directories
# image_dir = deeplesion_preprocessed_image_path
# label_dir = deeplesion_data_dir / "labels_unsorted"

# # Create .txt files
# label_dir.mkdir(parents=True, exist_ok=True)

# # Create target directory
# target_dir = deeplesion_data_dir / "deeplesion_yolo"
# if target_dir.exists() and target_dir.is_dir():
#     shutil.rmtree(target_dir)

# for idx, row in deeplesion_metadata_preprocessed.iterrows():
#     # Extract only images with annotated lesions (Val + Test)
#     if row["Train_Val_Test"] == 1:
#         continue

#     file_name = row["File_name"]
#     image_path = os.path.join(str(image_dir), file_name)
#     label_path = os.path.join(str(label_dir), file_name.replace(".png", ".txt"))

#     if not os.path.exists(image_path):
#         continue

#     lesion_type = row["Coarse_lesion_type"] - 1 # YOLO requires class IDs starting at 0
#     bbox_str = row["Bounding_boxes"]
#     size_str = row["Image_size"]

#     # Extract ground truth bounding box' coordinates
#     bbox_coords = [float(val) for val in bbox_str.split(",")]
#     x1, y1, x2, y2 = [round(c) for c in bbox_coords]

#     # Extract sizes
#     image_sizes = [int(val) for val in size_str.split(",")]
#     width, height = [size for size in image_sizes]

#     bbox_width = x2 - x1
#     bbox_height = y2 - y1
#     x_center = x1 + bbox_width / 2
#     y_center = y1 + bbox_height / 2

#     # Normalize
#     x_center /= width
#     y_center /= height
#     bbox_width /= width
#     bbox_height /= height

#     with open(label_path, 'a') as f:
#         f.write(f"{lesion_type} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}\n")

# # -----------------------------------------------------------------------------

# splits = ["train", "val", "test"]
# for split in splits:
#     (target_dir / "images" / split).mkdir(parents=True, exist_ok=True)
#     (target_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

# # Collect all images with annotated lesions
# annotated_images = [img for img in image_dir.glob("*.png") if (label_dir / (img.stem + ".txt")).exists()]
# random.seed(current_seed)
# random.shuffle(annotated_images)

# # Split into train, val and test sets
# n_total = len(annotated_images)
# n_train = int(0.7 * n_total)
# n_val = int(0.15 * n_total)

# train_images = annotated_images[:n_train]
# val_images = annotated_images[n_train:n_train + n_val]
# test_images = annotated_images[n_train + n_val:]

# splits_map = {
#     "train": train_images,
#     "val": val_images,
#     "test": test_images
# }

# # Copy image files
# for split, images in splits_map.items():
#     for image_path in images:
#         label_path = label_dir / (image_path.stem + ".txt")
#         shutil.copy(image_path, target_dir / "images" / split / image_path.name)
#         shutil.copy(label_path, target_dir / "labels" / split / label_path.name)

# print(f"Split done! Total = {n_total}")

# # Generate deeplesion.yaml
# dataset_root = os.path.abspath(str(target_dir))
# deeplesion_yaml = {
#     "path": dataset_root,
#     "train": os.path.join(dataset_root, "images/train"),
#     "val": os.path.join(dataset_root, "images/val"),
#     "test": os.path.join(dataset_root, "images/test"),
#     "nc": 8,
#     "names": [
#         "bone",
#         "abdomen",
#         "mediastinum",
#         "liver",
#         "lung",
#         "kidney",
#         "soft_tissue",
#         "pelvis"
#     ]
# }

# with open(target_dir / "deeplesion.yaml", "w") as f:
#     yaml.dump(deeplesion_yaml, f)

# # Remove directory with unsorted labels
# if label_dir.exists() and label_dir.is_dir():
#     shutil.rmtree(label_dir)


## Faster R-CNN format

In [ ]:
# deeplesion_metadata_preprocessed = load_metadata(deeplesion_preprocessed_metadata_path)

# # Source directories
# image_dir = deeplesion_preprocessed_image_path
# label_dir = deeplesion_data_dir / "labels_unsorted"

# # Create .txt files
# label_dir.mkdir(parents=True, exist_ok=True)

# # Create target directory
# target_dir = deeplesion_data_dir / "deeplesion_fasterrcnn_split_X"  # "X" is to be replaced with a number 1-3
# if target_dir.exists() and target_dir.is_dir():
#     shutil.rmtree(target_dir)

# for idx, row in deeplesion_metadata_preprocessed.iterrows():
#     # Extract only images with annotated lesions (Val + Test)
#     if row["Train_Val_Test"] == 1:
#         continue

#     file_name = row["File_name"]
#     image_path = os.path.join(str(image_dir), file_name)
#     label_path = os.path.join(str(label_dir), file_name.replace(".png", ".txt"))

#     if not os.path.exists(image_path):
#         continue

#     lesion_type = row["Coarse_lesion_type"] # Faster R-CNN uses the class 0 implicitly as the background class (no need for subtraction)
#     bbox_str = row["Bounding_boxes"]
#     size_str = row["Image_size"]

#     # Extract ground truth bounding box' coordinates
#     bbox_coords = [float(val) for val in bbox_str.split(",")]
#     x1, y1, x2, y2 = [round(c) for c in bbox_coords]

#     with open(label_path, 'a') as f:
#         f.write(f"{lesion_type} {x1} {y1} {x2} {y2}\n")

# # -----------------------------------------------------------------------------

# splits = ["train", "val", "test"]
# for split in splits:
#     (target_dir / "images" / split).mkdir(parents=True, exist_ok=True)
#     (target_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

# # Collect all images with annotated lesions
# annotated_images = [img for img in image_dir.glob("*.png") if (label_dir / (img.stem + ".txt")).exists()]
# random.seed(current_seed)
# random.shuffle(annotated_images)

# # Split into train, val and test sets
# n_total = len(annotated_images)
# n_train = int(0.7 * n_total)
# n_val = int(0.15 * n_total)

# train_images = annotated_images[:n_train]
# val_images = annotated_images[n_train:n_train + n_val]
# test_images = annotated_images[n_train + n_val:]

# splits_map = {
#     "train": train_images,
#     "val": val_images,
#     "test": test_images
# }

# # Copy image files
# for split, images in splits_map.items():
#     for image_path in images:
#         label_path = label_dir / (image_path.stem + ".txt")
#         shutil.copy(image_path, target_dir / "images" / split / image_path.name)
#         shutil.copy(label_path, target_dir / "labels" / split / label_path.name)

# print(f"Split done! Total = {n_total}")

# # Remove directory with unsorted labels
# if label_dir.exists() and label_dir.is_dir():
#     shutil.rmtree(label_dir)
